# 법인카드 전표 계정과목 AI 추천 실습
### 흐름: 더미데이터 → 파인튜닝 → Qdrant 저장 → 추천 테스트

> **순서대로 셀을 실행하세요** (Shift+Enter)

## STEP 1. 필요한 라이브러리 설치

In [ ]:
# 처음 한 번만 실행하면 됩니다 (2~3분 소요)
!pip install sentence-transformers qdrant-client datasets -q

## STEP 2. 더미 전표 데이터 생성

In [ ]:
# 더미 전표 데이터
# 형식: (전표 텍스트, 계정과목)

raw_data = [
    # 복리후생비
    ("스타벅스 강남점 / 접대비 / 45000원",         "복리후생비"),
    ("투썸플레이스 판교점 / 회식 / 38000원",        "복리후생비"),
    ("할리스커피 서초점 / 팀회식 / 52000원",        "복리후생비"),
    ("파리바게뜨 역삼점 / 간식구매 / 23000원",      "복리후생비"),
    ("맥도날드 삼성점 / 팀점심 / 67000원",          "복리후생비"),
    ("배스킨라빈스 선릉점 / 간식 / 31000원",        "복리후생비"),
    ("CGV 강남 / 팀문화행사 / 88000원",             "복리후생비"),

    # 접대비
    ("한우마을 청담 / 거래처접대 / 320000원",       "접대비"),
    ("노부레스토랑 강남 / 고객접대 / 450000원",     "접대비"),
    ("스시오마카세 서울 / 클라이언트미팅 / 280000원", "접대비"),
    ("리츠칼튼 라운지 / 거래처식사 / 195000원",     "접대비"),
    ("파크하얏트 바 / 고객접대 / 380000원",         "접대비"),

    # 차량유지비
    ("현대오일뱅크 서초 / 주유 / 80000원",          "차량유지비"),
    ("SK주유소 판교 / 연료충전 / 75000원",           "차량유지비"),
    ("GS칼텍스 강남 / 주유 / 92000원",              "차량유지비"),
    ("현대자동차서비스 / 차량점검 / 150000원",       "차량유지비"),
    ("한국주차장 강남 / 주차료 / 15000원",           "차량유지비"),
    ("KT하이웨이 / 고속도로통행료 / 12000원",        "차량유지비"),

    # 도서인쇄비
    ("YES24 / 기술서적구매 / 34000원",              "도서인쇄비"),
    ("교보문고 강남 / 업무서적 / 27000원",           "도서인쇄비"),
    ("알라딘 / 도서구매 / 19000원",                 "도서인쇄비"),
    ("인터파크도서 / 전문서적 / 42000원",            "도서인쇄비"),
    ("영풍문고 / 참고서적 / 23000원",               "도서인쇄비"),

    # 소모품비
    ("오피스디포 / 사무용품구매 / 56000원",          "소모품비"),
    ("다이소 강남 / 비품구매 / 18000원",             "소모품비"),
    ("문구야 / 사무용품 / 34000원",                 "소모품비"),
    ("이마트 / 사무실청소용품 / 45000원",            "소모품비"),
    ("쿠팡 / 프린터용지구매 / 29000원",              "소모품비"),

    # 통신비
    ("KT / 인터넷요금 / 110000원",                  "통신비"),
    ("SKT / 법인휴대폰요금 / 85000원",              "통신비"),
    ("LG유플러스 / 회선요금 / 95000원",             "통신비"),
    ("SK브로드밴드 / 사무실인터넷 / 77000원",        "통신비"),
]

print(f"총 {len(raw_data)}개 더미 전표 데이터 생성 완료")
print("\n샘플 3개:")
for text, label in raw_data[:3]:
    print(f"  입력: {text}")
    print(f"  정답: {label}\n")

## STEP 3. 파인튜닝용 데이터셋 구성 (유사/비유사 쌍)

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
import random

# 파인튜닝은 "이 둘은 같은 계정과목이다(유사)" vs "이 둘은 다르다(비유사)" 쌍으로 학습
train_examples = []

# 같은 계정과목끼리 → 유사 쌍 (label=1.0)
from itertools import combinations
grouped = {}
for text, label in raw_data:
    grouped.setdefault(label, []).append(text)

for label, texts in grouped.items():
    for t1, t2 in combinations(texts, 2):
        train_examples.append(InputExample(texts=[t1, t2], label=1.0))

# 다른 계정과목끼리 → 비유사 쌍 (label=0.0)
labels = list(grouped.keys())
for i in range(len(train_examples)):  # 유사 쌍 수만큼 비유사 쌍도 생성
    l1, l2 = random.sample(labels, 2)
    t1 = random.choice(grouped[l1])
    t2 = random.choice(grouped[l2])
    train_examples.append(InputExample(texts=[t1, t2], label=0.0))

random.shuffle(train_examples)
print(f"학습 데이터 쌍: {len(train_examples)}개")
print(f"  유사 쌍 (같은 계정): {len(train_examples)//2}개")
print(f"  비유사 쌍 (다른 계정): {len(train_examples)//2}개")

## STEP 4. 임베딩 모델 파인튜닝

In [ ]:
from sentence_transformers import SentenceTransformer, losses

# 베이스 모델 로드 (한국어 지원 경량 모델)
# bge-m3는 너무 크므로 실습용으로 paraphrase-multilingual 사용
print("모델 로딩 중...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("모델 로딩 완료!")

# DataLoader 구성
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# 손실함수: CosineSimilarityLoss
# → 유사 쌍은 코사인 유사도 1에 가깝게, 비유사 쌍은 0에 가깝게 학습
train_loss = losses.CosineSimilarityLoss(model)

# 파인튜닝 실행 (epoch=5, 약 1~2분 소요)
print("\n파인튜닝 시작...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    warmup_steps=10,
    show_progress_bar=True
)

# 파인튜닝된 모델 저장
model.save('./finetuned_model')
print("\n파인튜닝 완료! 모델 저장됨 → ./finetuned_model")

## STEP 5. Qdrant에 전표 데이터 저장

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Qdrant 인메모리 모드로 실행 (설치 없이 바로 사용 가능)
client = QdrantClient(":memory:")

# 컬렉션 생성
COLLECTION_NAME = "전표"
VECTOR_SIZE = model.get_sentence_embedding_dimension()

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
)
print(f"컬렉션 생성 완료 (벡터 차원: {VECTOR_SIZE})")

# 전표 데이터 임베딩 후 Qdrant에 저장
points = []
for idx, (text, label) in enumerate(raw_data):
    vector = model.encode(text).tolist()
    points.append(PointStruct(
        id=idx,
        vector=vector,
        payload={"전표내용": text, "계정과목": label}
    ))

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Qdrant에 {len(points)}개 전표 저장 완료!")

## STEP 6. 계정과목 추천 테스트 🎯

In [ ]:
def 계정과목_추천(query_text, top_k=3):
    query_vector = model.encode(query_text).tolist()
    
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k
    ).points
    
    print(f"\n📋 입력 전표: {query_text}")
    print("─" * 60)
    print(f"{'유사 전표':<35} {'계정과목':<12} {'유사도'}")
    print("─" * 60)
    
    account_votes = {}
    for hit in results:
        전표 = hit.payload['전표내용']
        계정 = hit.payload['계정과목']
        유사도 = hit.score
        print(f"{전표[:33]:<35} {계정:<12} {유사도:.4f}")
        account_votes[계정] = account_votes.get(계정, 0) + 유사도
    
    추천계정 = max(account_votes, key=account_votes.get)
    print("─" * 60)
    print(f"\n✅ 추천 계정과목: [{추천계정}]")
    return 추천계정


계정과목_추천("이디야커피 잠실점 / 팀회식 / 41000원")

print("\n" + "="*60 + "\n")

계정과목_추천("에쓰오일 판교점 / 주유 / 68000원")

print("\n" + "="*60 + "\n")

계정과목_추천("반디앤루니스 / 업무관련서적 / 31000원")

## 직접 테스트해보세요!
아래 셀의 `query` 값을 바꿔서 원하는 전표를 입력해보세요.

In [ ]:
# 여기에 원하는 전표 내용 입력!
query = "롯데렌터카 / 출장차량대여 / 120000원"

계정과목_추천(query)